# Prédiction du Churn Client Bancaire
## Analyse de Données d'une banque  
*Projet ML/PI — ESPRIT School of Business 2025/2026

In [ ]:
# ============================================
# CELLULE 1 : IMPORTS + CHARGEMENT
# ============================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Chargement du fichier principal
df = pd.read_csv(
    r'C:\Users\JIHEN\Desktop\M1 BA\ml projet\data_churn (1).csv', 
    encoding='utf-8'
)

print("✅ Fichier chargé avec succès !")
print(f"📊 Dimensions : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

## Task 1 - Chargement des Données

In [ ]:

df.head()

In [ ]:
# ============================================
# CELLULE 3 : TYPES ET VALEURS MANQUANTES
# ============================================
df.info()

## Task 2 - Exploration des Données (EDA)
### ➤ Structure et Typologie des Variables

In [ ]:
# ============================================
# CELLULE 4 : STATISTIQUES DESCRIPTIVES
# ============================================
df.describe()

In [ ]:
# ============================================
# CELLULE 5 : STATISTIQUES VARIABLES TEXTE
# (comme le prof — describe(include=['object']))
# ============================================
df.describe(include=['object'])

### ➤ Vérification des Valeurs Manquantes

In [ ]:
# ============================================
# CELLULE 6 : VALEURS MANQUANTES
# ============================================
manquants = pd.DataFrame({
    'Colonne': df.columns,
    'Nb_Manquants': df.isnull().sum().values,
    'Pourcentage_%': (df.isnull().sum().values / len(df) * 100).round(2)
})

manquants = manquants[manquants['Nb_Manquants'] > 0].sort_values(
    'Pourcentage_%', ascending=False
).reset_index(drop=True)

print("=== VALEURS MANQUANTES ===")
print(manquants.to_string(index=False))

In [ ]:
# ============================================
# CELLULE 7 : VISUALISATION VALEURS MANQUANTES
# ============================================
plt.figure(figsize=(12, 6))
plt.bar(manquants['Colonne'], manquants['Pourcentage_%'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.ylabel('% Valeurs Manquantes')
plt.title('Pourcentage de valeurs manquantes par colonne')
plt.tight_layout()
plt.show()

### ➤ Justification Exclusion Corporate

In [ ]:
# ============================================
# CELLULE 9 : ANALYSE PARTYCLASS
# Justifier l'exclusion Corporate
# ============================================
print("=== RÉPARTITION PAR SEGMENT (niveau ligne) ===")
print(df['PARTYCLASS'].value_counts())
print()
print(df['PARTYCLASS'].value_counts(normalize=True).round(4) * 100)

# Visualisation
plt.figure(figsize=(8, 5))
df['PARTYCLASS'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Répartition des segments clients')
plt.ylabel('Nombre de lignes')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 10 : PARTYCLASS AU NIVEAU CLIENT
# (Le vrai chiffre qui guide la décision)
# ============================================
clients_par_type = df.groupby('CUSTOMER_NO')['PARTYCLASS'].first()
print("=== RÉPARTITION PAR SEGMENT (niveau client unique) ===")
print(clients_par_type.value_counts())
print()
print(clients_par_type.value_counts(normalize=True).round(4) * 100)

In [ ]:
# ============================================
# CELLULE 11 : PREUVE DU CLIENT ANORMAL
# Client C155581 à 8091 comptes
# ============================================
nb_comptes = df.groupby('CUSTOMER_NO')['ACCOUNT_NO'].count()

print("=== DISTRIBUTION NB COMPTES PAR CLIENT ===")
print(nb_comptes.describe())
print()
print("Top 5 clients avec le plus de comptes :")
print(nb_comptes.sort_values(ascending=False).head(5))

In [ ]:
# ============================================
# CELLULE 12 : PROFIL DU CLIENT SUSPECT
# Confirmer que c'est un Corporate
# ============================================
client_max = nb_comptes.idxmax()
df_suspect = df[df['CUSTOMER_NO'] == client_max]

print(f"Client : {client_max} → {nb_comptes.max()} comptes")
print()
print(df_suspect[['PARTYCLASS', 'LOB', 'BRANCH', 
                   'PRODUCT_LINE']].value_counts().head(5))

In [ ]:
# ============================================
# CELLULE 13 : SALARY SELON PARTYCLASS
# Preuve que Corporate n'a pas de salaire
# ============================================
print("=== TAUX DE MANQUANTS SALARY PAR SEGMENT ===")
print(df.groupby('PARTYCLASS')['SALARY'].apply(
    lambda x: round(x.isnull().mean() * 100, 2)
).sort_values(ascending=False))

### ➤ Variable Cible - ACCOUNT_STATUS

In [ ]:
# ============================================
# CELLULE 14 : ACCOUNT_STATUS - VARIABLE CIBLE
# Analyser avant filtrage
# ============================================
print("=== ACCOUNT_STATUS (niveau compte) ===")
print(df['ACCOUNT_STATUS'].value_counts())
print()
print(df['ACCOUNT_STATUS'].value_counts(normalize=True).round(3) * 100)

# Visualisation
plt.figure(figsize=(6, 5))
df['ACCOUNT_STATUS'].value_counts().plot(kind='bar', 
                                          color=['#2196F3', '#FF5722'],
                                          edgecolor='white')
plt.title('Répartition ACCOUNT_STATUS (niveau compte)')
plt.ylabel('Nombre de comptes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### ➤ Filtrage Périmètre Retail + Elite

In [ ]:
# ============================================
# CELLULE 15 : FILTRAGE PÉRIMÈTRE FINAL
# Retail + Elite UNIQUEMENT
# Corporate Small EXCLU
# ============================================
segments_a_garder = ['Retail', 'Elite']
df_clean = df[df['PARTYCLASS'].isin(segments_a_garder)].copy()
df_clean = df_clean.dropna(subset=['ACCOUNT_STATUS'])

print(f"✅ Avant filtrage  : {len(df):,} lignes")
print(f"✅ Après filtrage  : {len(df_clean):,} lignes")
print(f"🗑️  Supprimées      : {len(df) - len(df_clean):,} lignes")
print(f"👥 Clients uniques : {df_clean['CUSTOMER_NO'].nunique():,}")
print()
print("Répartition finale :")
print(df_clean['PARTYCLASS'].value_counts())

In [ ]:
# ============================================
# CELLULE 16 : VARIABLE CIBLE AU NIVEAU CLIENT
# Churn = 1 si TOUS les comptes sont Closed
# ============================================
churn_client = df_clean.groupby('CUSTOMER_NO')['ACCOUNT_STATUS'].apply(
    lambda x: 1 if (x == 'Closed').all() else 0
).reset_index()
churn_client.columns = ['CUSTOMER_NO', 'CHURN']

print("=== TAUX DE CHURN AU NIVEAU CLIENT ===")
print(churn_client['CHURN'].value_counts())
print()
print(churn_client['CHURN'].value_counts(normalize=True).round(3) * 100)

# Camembert (comme rapport amie)
plt.figure(figsize=(6, 6))
sizes = churn_client['CHURN'].value_counts().values
labels = [f'Non-Churn (0)\n{sizes[0]:,}', f'Churn (1)\n{sizes[1]:,}']
colors = ['#2196F3', '#FF5722']
plt.pie(sizes, labels=labels, colors=colors, 
        autopct='%1.1f%%', startangle=90)
plt.title('Distribution du Churn au niveau Client')
plt.tight_layout()
plt.show()

## Task 3 - Analyse Univariée
### ➤ Variables Catégorielles

In [ ]:
# ============================================
# CELLULE 17 : ANALYSE UNIVARIÉE — SCORE KYC
# Variable catégorielle importante
# ============================================
print("=== SCORE KYC ===")
print(df_clean['SCORE_KYC'].value_counts())
print()

plt.figure(figsize=(7, 5))
df_clean['SCORE_KYC'].value_counts().plot(kind='bar', 
                                           color='steelblue',
                                           edgecolor='white')
plt.title('Répartition du Score KYC')
plt.ylabel('Nombre de lignes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 18 : ANALYSE UNIVARIÉE — LOB / SEGMENT
# ============================================
print("=== SEGMENT CLIENT (LOB) ===")
lob_map = {1: 'Private', 3: 'Retail Affluent', 
           4: 'Retail', 11: 'Small Business', 
           2: 'Elite', 13: 'Corporate Large'}
df_clean['LOB_DESC'] = df_clean['LOB'].map(lob_map).fillna('Autre')

print(df_clean['LOB_DESC'].value_counts())
print()

plt.figure(figsize=(8, 5))
df_clean['LOB_DESC'].value_counts().plot(kind='bar',
                                          color='steelblue',
                                          edgecolor='white')
plt.title('Répartition par Segment Client (LOB)')
plt.ylabel('Nombre de lignes')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 19 : ANALYSE UNIVARIÉE — PRODUCT_LINE
# ============================================
print("=== PRODUCT_LINE ===")
print(df_clean['PRODUCT_LINE'].value_counts())
print()

plt.figure(figsize=(7, 5))
df_clean['PRODUCT_LINE'].value_counts().plot(kind='bar',
                                              color='steelblue',
                                              edgecolor='white')
plt.title('Répartition par Ligne de Produit')
plt.ylabel('Nombre de comptes')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

 ### ➤ Variables Numériques

In [ ]:
# ============================================
# CELLULE 20 : ANALYSE UNIVARIÉE — SALARY
# Distribution + outliers
# ============================================
salary_clean = df_clean['SALARY'].dropna()

print("=== STATISTIQUES SALARY ===")
print(salary_clean.describe())
print(f"\nNb clients avec salary = 0 : {(salary_clean == 0).sum():,}")
print(f"Nb clients avec salary > 0 : {(salary_clean > 0).sum():,}")

# Boxplot 
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.boxplot(data=salary_clean, orient='h', color='steelblue')
plt.title('Boxplot SALARY')
plt.xlabel('Salaire (TND)')

plt.subplot(1, 2, 2)
salary_clean[salary_clean < 10000].hist(bins=50, color='steelblue', 
                                         edgecolor='white')
plt.title('Distribution SALARY (< 10 000 TND)')
plt.xlabel('Salaire (TND)')
plt.ylabel('Fréquence')
plt.tight_layout()
plt.show()

 ### ➤ Valeurs Aberrantes - Méthode IQR

In [ ]:
# ============================================
# CELLULE 21 : IQR SALARY

# ============================================
Q1 = salary_clean.quantile(0.25)
Q3 = salary_clean.quantile(0.75)
IQR = Q3 - Q1
Inf = Q1 - 1.5 * IQR
Sup = Q3 + 1.5 * IQR

print("=== IQR SALARY ===")
print(f"Q1      : {Q1:,.0f} TND")
print(f"Q3      : {Q3:,.0f} TND")
print(f"IQR     : {IQR:,.0f} TND")
print(f"Borne inférieure : {Inf:,.0f} TND")
print(f"Borne supérieure : {Sup:,.0f} TND")
print()

outliers_inf = (salary_clean < Inf).sum()
outliers_sup = (salary_clean > Sup).sum()
print(f"Outliers bas  (< {Inf:,.0f}) : {outliers_inf:,}")
print(f"Outliers hauts (> {Sup:,.0f}) : {outliers_sup:,}")
print(f"Total outliers : {outliers_inf + outliers_sup:,}")
print(f"% outliers : {(outliers_inf + outliers_sup)/len(salary_clean)*100:.2f}%")

In [ ]:
# ============================================
# CELLULE 22 : IQR ACCT_BALANCE
# ============================================
balance_clean = df_clean['ACCT_BALANCE'].dropna()

Q1b = balance_clean.quantile(0.25)
Q3b = balance_clean.quantile(0.75)
IQRb = Q3b - Q1b
Infb = Q1b - 1.5 * IQRb
Supb = Q3b + 1.5 * IQRb

print("=== IQR ACCT_BALANCE ===")
print(f"Q1      : {Q1b:,.2f}")
print(f"Q3      : {Q3b:,.2f}")
print(f"IQR     : {IQRb:,.2f}")
print(f"Borne inférieure : {Infb:,.2f}")
print(f"Borne supérieure : {Supb:,.2f}")
print()

outliers_b = ((balance_clean < Infb) | (balance_clean > Supb)).sum()
print(f"Total outliers : {outliers_b:,}")
print(f"% outliers : {outliers_b/len(balance_clean)*100:.2f}%")

# Boxplot ACCT_BALANCE
plt.figure(figsize=(8, 4))
sns.boxplot(data=balance_clean, orient='h', color='steelblue')
plt.title('Boxplot ACCT_BALANCE')
plt.tight_layout()
plt.show()

### ➤ Outliers après Filtrage

In [ ]:
# Qui sont les outliers salary APRÈS exclusion Corporate ?
salary_outliers = df_clean[df_clean['SALARY'] > 3518]

print(f"Nb clients outliers salary : {salary_outliers['CUSTOMER_NO'].nunique():,}")
print()
print("Répartition par segment :")
print(salary_outliers['PARTYCLASS'].value_counts())
print()
print("Répartition par LOB :")
print(salary_outliers['LOB'].value_counts())
print()
print("Stats salary outliers :")
print(salary_outliers['SALARY'].describe())

In [ ]:
# CELLULE 31 — % Outliers par colonne
cols_num = ['SALARY', 'ACCT_BALANCE', 'AMOUNT', 'FIXEDRATE']
outliers_pct = {}
for col in cols_num:
    s = df_clean[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    pct = ((s < Q1-1.5*IQR) | (s > Q3+1.5*IQR)).mean()*100
    outliers_pct[col] = round(pct, 2)
    print(f"{col} : {pct:.2f}%")

plt.figure(figsize=(8,5))
plt.bar(outliers_pct.keys(), outliers_pct.values(),
        color='steelblue', edgecolor='white')
plt.title('Pourcentage de Valeurs Aberrantes par Colonne (IQR)')
plt.ylabel('% Outliers')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 23 : BOXPLOT MULTI-VARIABLES
# ============================================
# Variables numériques exploitables directement
num_vars = ['SALARY', 'ACCT_BALANCE', 'AMOUNT', 'FIXEDRATE']
df_num = df_clean[num_vars].dropna()

plt.figure(figsize=(10, 5))
sns.boxplot(data=df_num[['SALARY', 'ACCT_BALANCE']], orient='h')
plt.title('Boxplot SALARY vs ACCT_BALANCE')
plt.tight_layout()
plt.show()

### ➤ Asymétrie et Aplatissement

In [ ]:
# ============================================
# CELLULE 24 : SKEWNESS ET KURTOSIS

# ============================================
num_cols = ['SALARY', 'ACCT_BALANCE', 'AMOUNT', 'FIXEDRATE']
stats_sk = pd.DataFrame({
    'Variable': num_cols,
    'Skewness': [df_clean[c].skew() for c in num_cols],
    'Kurtosis': [df_clean[c].kurtosis() for c in num_cols]
}).round(2)

print("=== ASYMÉTRIE ET APLATISSEMENT ===")
print(stats_sk.to_string(index=False))

### ➤ Scatter Plot - Âge vs Salaire

In [ ]:
# ============================================
# CELLULE 25 : SCATTER PLOT — Age vs Salary
# DATE_OF_BIRTH = année brute float (1969.0)
# ============================================

# Recréer AGE directement depuis df (avant conversion datetime)
df_clean['AGE'] = 2025 - df.loc[df_clean.index, 'DATE_OF_BIRTH']

scatter_data = df_clean[['AGE', 'SALARY']].dropna()
scatter_data = scatter_data[
    (scatter_data['AGE'] >= 18) & 
    (scatter_data['AGE'] <= 80) &
    (scatter_data['SALARY'] < 20000) &
    (scatter_data['SALARY'] > 0)
]

print(f"Points affichés : {len(scatter_data):,}")

plt.figure(figsize=(8, 5))
plt.scatter(scatter_data['AGE'], scatter_data['SALARY'],
            alpha=0.3, s=5, color='steelblue')
plt.xlabel('Âge')
plt.ylabel('Salaire (TND)')
plt.title('Scatter Plot — Âge vs Salaire')
plt.tight_layout()
plt.show()

## Task 4 - Analyse Multivariée
### ➤ Churn par Segment et Score KYC

In [ ]:
# ============================================
# CELLULE 26 : ANALYSE BIVARIÉE
# Churn par Segment (LOB)
# ============================================
df_avec_churn = df_clean.merge(churn_client, on='CUSTOMER_NO', how='left')

churn_par_lob = df_avec_churn.groupby('LOB_DESC')['CHURN'].mean().round(3) * 100
churn_par_lob = churn_par_lob.sort_values(ascending=False)

print("=== TAUX DE CHURN PAR SEGMENT ===")
print(churn_par_lob)

plt.figure(figsize=(8, 5))
churn_par_lob.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Taux de Churn par Segment Client')
plt.ylabel('Taux de Churn (%)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# CELLULE 27 : CHURN PAR SCORE KYC
#  insight métier bancaire
# ============================================
churn_par_kyc = df_avec_churn.groupby('SCORE_KYC')['CHURN'].mean().round(3) * 100
churn_par_kyc = churn_par_kyc.sort_values(ascending=False)

print("=== TAUX DE CHURN PAR SCORE KYC ===")
print(churn_par_kyc)

plt.figure(figsize=(7, 5))
churn_par_kyc.plot(kind='bar', 
                   color=['#FF5722','#FF9800','#FFC107','#4CAF50'],
                   edgecolor='white')
plt.title('Taux de Churn par Score KYC')
plt.ylabel('Taux de Churn (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### ➤ Corrélation — Heatmap

In [ ]:
# ============================================
# CELLULE 28 : HEATMAP CORRÉLATION

# ============================================
cols_corr = ['SALARY', 'ACCT_BALANCE', 'AMOUNT', 'FIXEDRATE', 'LOB']
corr_data = df_clean[cols_corr].dropna()
corr_matrix = corr_data.corr().round(2)

print("=== MATRICE DE CORRÉLATION ===")
print(corr_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm',
            center=0, fmt='.2f', linewidths=0.5)
plt.title('Matrice de Corrélation — Variables Numériques')
plt.tight_layout()
plt.show()

### ➤ Pairplot

In [ ]:
# ============================================
# CELLULE 29 : PAIRPLOT

# ============================================
pairplot_data = df_clean[['SALARY', 'ACCT_BALANCE',
                           'AMOUNT', 'FIXEDRATE']].dropna()
pairplot_data = pairplot_data.sample(2000, random_state=42)

sns.set(style='ticks')
sns.pairplot(pairplot_data)
plt.suptitle('Pairplot — Variables Numériques', y=1.02)
plt.show()

In [ ]:
# ============================================
# VÉRIFICATION AGES ABERRANTS
# ============================================
df['AGE_CHECK'] = 2025 - df['DATE_OF_BIRTH'].astype(str)\

print("=== DISTRIBUTION DES AGES ===")
print(df['AGE_CHECK'].describe())
print()
print(f"Clients < 18 ans  : {(df['AGE_CHECK'] < 18).sum():,}")
print(f"Clients > 80 ans  : {(df['AGE_CHECK'] > 80).sum():,}")
print(f"Clients 18-80 ans : {((df['AGE_CHECK'] >= 18) & (df['AGE_CHECK'] <= 80)).sum():,}")
print(f"AGE manquant      : {df['AGE_CHECK'].isna().sum():,}")
print()
print("Ages aberrants < 18 :")
print(df[df['AGE_CHECK'] < 18]['AGE_CHECK'].value_counts().head(10))
print()
print("Ages aberrants > 80 :")
print(df[df['AGE_CHECK'] > 80]['AGE_CHECK'].value_counts().head(10))

## Task 5 — Résumé EDA

In [ ]:
# ============================================
# CELLULE 30 : RÉSUMÉ EDA FINAL
# ============================================
print("=" * 55)
print("       RÉSUMÉ EDA — DÉCISIONS PRISES")
print("=" * 55)
print(f"Dataset initial        : {len(df):,} lignes")
print(f"Après filtrage         : {len(df_clean):,} lignes")
print(f"Clients uniques        : {df_clean['CUSTOMER_NO'].nunique():,}")
print()
print("PÉRIMÈTRE RETENU :")
print("  → Retail + Elite uniquement")
print("  → Corporate exclu (features incompatibles)")
print()
print("FUITES DE DONNÉES EXCLUES DU ML :")
print("  → ACCT_CLOSE_DATE   : postérieure au churn")
print("  → CLOSURE_REASON    : postérieure au churn")
print("  → PRODUCT_STATUS    : signal direct du churn")
print()
print("VARIABLES À TRAITER :")
print("  → SALARY (61.76%)   : régression d'imputation")
print("  → DATE_OF_BIRTH     : année brute → AGE = 2025 - année")
print("  → ACCOUNT_CATEGORY  : jointure dim_CATEGORY")
print()
print("OUTLIERS DÉTECTÉS :")
print("  → SALARY   : valeurs > 100 000 TND → IQR")
print("  → AMOUNT   : valeurs négatives et extrêmes")
print("  → ACCT_BALANCE : valeurs négatives (crédits normaux)")

In [ ]:
# ============================================
# DÉFINITION FONCTION + EXPORT AVANT TALEND
# ============================================
import unicodedata

def enlever_accents(texte):
    if pd.isna(texte): return ""
    texte = str(texte)
    texte = unicodedata.normalize('NFKD', texte).encode(
        'ASCII', 'ignore'
    ).decode('ASCII')
    return texte.upper()

def categoriser_compte(texte):
    if pd.isna(texte): return "AUTRE"
    t = enlever_accents(texte)
    if 'DECEDE' in t or 'INDISP' in t: return "BLOQUE"
    if ('EXIGIBLE' in t or 'LELLA' in t or
        'HUISS' in t or 'CTX' in t or
        'RECOUVREMENT' in t or 'IMPAYE' in t or
        'ENGAG EN PPL' in t): return "TECHNIQUE"
    if ('EPARGNE' in t or 'COURANT' in t or
        'VUE' in t or 'ALLOCATION' in t): return "DAV"
    if 'CREDIT' in t or 'TEGF' in t: return "CREDIT"
    if ('TERME' in t or 'BON DE CAISSE' in t or
        'DEVISES' in t or 'CONVERTIBLE' in t or
        'TOURISTIQUE' in t or 'PPR' in t or
        'PARTICIPATIF' in t or 'SAFE DEPOSIT' in t): return "PLACEMENT"
    return "AUTRE"

# MACRO_CATEGORIE
df['MACRO_CATEGORIE'] = df['ACCOUNT_TYPE_DESC'].apply(categoriser_compte)

# Formats dates
df['CUST_OPENING_DATE_CLEAN'] = pd.to_datetime(
    df['CUST_OPENING_DATE'].astype(str).str.split('.').str[0],
    format='%Y%m%d', errors='coerce'
).dt.strftime('%Y-%m-%d')

df['DATE_OF_BIRTH_CLEAN'] = df['DATE_OF_BIRTH'].astype(str)\
    .str.split('.').str[0].replace('nan', '')

df['ACCT_OPENING_DATE_CLEAN'] = pd.to_datetime(
    df['ACCT_OPENING_DATE'].astype(str).str.split('.').str[0],
    format='%Y%m%d', errors='coerce'
).dt.strftime('%Y-%m-%d')

# Export
colonnes_export = [
    'CUSTOMER_NO', 'ACCOUNT_NO', 'PARTYCLASS', 'LOB',
    'ACCOUNT_STATUS', 'ACCT_BALANCE', 'SALARY',
    'PRODUCT_LINE', 'AMOUNT', 'FIXEDRATE',
    'BRANCH', 'NATIONALITY', 'RESIDENCE',
    'SCORE_KYC', 'INDUSTRY',
    'CUST_OPENING_DATE_CLEAN',
    'DATE_OF_BIRTH_CLEAN',
    'ACCT_OPENING_DATE_CLEAN',
    'MACRO_CATEGORIE',
    'ACCOUNT_TYPE_DESC'
]

df[colonnes_export].to_csv(
    r'C:\Users\JIHEN\Desktop\M1 BA\ml projet\stg_churn_macro.csv',
    index=False, encoding='utf-8'
)

print(f"✅ Export terminé : {len(df):,} lignes")
print()
print("Vérification dates :")
print(df[['CUST_OPENING_DATE_CLEAN',
          'DATE_OF_BIRTH_CLEAN',
          'MACRO_CATEGORIE']].head(5))